## Mini Case — Análise de Vendas (E-commerce)

### Contexto

Você atua como **Data Analyst / Data Engineer** em um e-commerce.

O time de negócios solicitou uma análise exploratória para entender melhor o desempenho das vendas e apoiar decisões estratégicas.

---

### Dataset

Você recebeu o seguinte dataset para análise:

- **Arquivo:** `dados_vendas.csv`
- **Descrição:** Contém informações de pedidos realizados em um período recente. 

---

### Objetivo

Utilizar ferramentas de análise de dados (como **Spark, Pandas ou Polars**) para explorar o dataset e gerar insights relevantes para o negócio.

In [45]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import sum, avg, desc, count, month, to_date, col

spark = SparkSession.builder.appName('Projeto1').getOrCreate()

In [6]:
df = spark.read.format('csv') \
    .option('header', 'true') \
    .option('inferSchema', 'true') \
    .load('/home/jovyan/estudos_spark/arquivos/dados_vendas.csv')

In [8]:
df.show()

+---------+----------+--------+----------+------+-----+
|id_pedido|      data| produto| categoria|cidade|valor|
+---------+----------+--------+----------+------+-----+
|        1|2024-01-10|Notebook|Eletronico|    SP| 3500|
|        2|2024-01-12| Celular|Eletronico|    RJ| 2200|
|        3|2024-01-15|Camiseta|     Roupa|    SP|  120|
|        4|2024-02-01|   Calça|     Roupa|    MG|  200|
|        5|2024-02-10|    Fone|Eletronico|    SP|  300|
|        6|2024-02-15|   Tênis|   Calçado|    RJ|  450|
|        7|2024-03-05|Notebook|Eletronico|    MG| 3700|
|        8|2024-03-10|Camiseta|     Roupa|    SP|   90|
|        9|2024-03-20| Celular|Eletronico|    SP| 2500|
|       10|2024-03-25|   Tênis|   Calçado|    MG|  500|
+---------+----------+--------+----------+------+-----+



## Parte 1 — Exploração

### Objetivo

Realizar uma análise inicial do dataset para compreender métricas básicas de negócio.

---

### Perguntas

1. Quantos pedidos existem no dataset?
2. Qual é o faturamento total?
3. Qual é o ticket médio?

---

### Competências avaliadas

- Uso de funções de agregação:
  - `count`
  - `sum`
  - `avg`
- Capacidade de exploração inicial de dados
- Interpretação de métricas básicas de negócio

In [15]:
count = df.count()
print(f'Quantos pedidos existem no dataset?: {count}')

Quantos pedidos existem no dataset?: 10


In [23]:
faturamento = df.agg(sum('valor').alias('faturamento'))
faturamento.show()

+-----------+
|faturamento|
+-----------+
|      13560|
+-----------+



In [26]:
ticket_medio = df.agg(avg('valor').alias('ticket_medio'))
ticket_medio.show()

+------------+
|ticket_medio|
+------------+
|      1356.0|
+------------+



## Parte 2 — Análise por Dimensão

### Objetivo

Analisar o comportamento das vendas a partir de diferentes dimensões, como categoria e cidade.

---

### Perguntas

1. Qual é o faturamento por categoria?
2. Qual é a categoria mais lucrativa?
3. Qual é o faturamento por cidade?

---

### Competências avaliadas

- Agrupamento de dados:
  - `groupBy` (Spark / Polars)
  - `groupby` (Pandas)
- Uso de funções de agregação
- Interpretação de métricas por dimensão

In [28]:
faturamento_categoria = df.groupBy('categoria').agg(sum('valor').alias('total'))
faturamento_categoria.show()

+----------+-----+
| categoria|total|
+----------+-----+
|Eletronico|12200|
|   Calçado|  950|
|     Roupa|  410|
+----------+-----+



In [32]:
categoria_top = faturamento_categoria.orderBy(desc('total')).first()
print(categoria_top)

Row(categoria='Eletronico', total=12200)


In [33]:
faturamento_cidade = df.groupBy('cidade').agg(sum('valor').alias('total'))
faturamento_cidade.show()

+------+-----+
|cidade|total|
+------+-----+
|    SP| 6510|
|    MG| 4400|
|    RJ| 2650|
+------+-----+



## Parte 3 — Produtos

### Objetivo

Analisar o desempenho dos produtos considerando volume de vendas e faturamento.

---

### Perguntas

1. Qual é o produto mais vendido em quantidade?
2. Qual é o produto que mais faturou?

---

### Competências avaliadas

- Uso de funções de agregação:
  - `count` para volume de vendas
  - `sum` para faturamento
- Compreensão da diferença entre métricas de quantidade e receita
- Capacidade de interpretar resultados sob diferentes perspectivas de negócio

In [41]:
produto_qtd = (
    df.groupBy('produto')
    .agg(count('*').alias('quantidade'))
    .orderBy(desc('quantidade'))
)

produto_qtd.show()

produto_mais_vendido = produto_qtd.first()
print(produto_mais_vendido)

+--------+----------+
| produto|quantidade|
+--------+----------+
| Celular|         2|
|Notebook|         2|
|Camiseta|         2|
|   Tênis|         2|
|    Fone|         1|
|   Calça|         1|
+--------+----------+

Row(produto='Celular', quantidade=2)


In [42]:
produto_faturamento = (
    df.groupBy('produto')
    .agg(sum('valor').alias('total_faturado'))
    .orderBy(desc('total_faturado'))
)

produto_faturamento.show()

produto_top_faturamento = produto_faturamento.first()
print(produto_top_faturamento)

+--------+--------------+
| produto|total_faturado|
+--------+--------------+
|Notebook|          7200|
| Celular|          4700|
|   Tênis|           950|
|    Fone|           300|
|Camiseta|           210|
|   Calça|           200|
+--------+--------------+

Row(produto='Notebook', total_faturado=7200)


## Parte 4 — Análise Temporal

### Objetivo

Analisar o comportamento das vendas ao longo do tempo, identificando padrões e variações no faturamento.

---

### Perguntas

1. Qual é o faturamento por mês?
2. Existe crescimento de janeiro para março?

**Resposta:**

Não há crescimento contínuo no período. O faturamento caiu significativamente de janeiro para fevereiro, mas apresentou forte recuperação em março.

---

### Competências avaliadas

- Manipulação de datas
- Extração de componentes temporais (mês)
- Agregação temporal
- Interpretação de tendências ao longo do tempo

In [44]:
df = df.withColumn('data', to_date('data'))

faturamento_mes = (
    df.withColumn('mes', month('data'))
    .groupBy('mes')
    .agg(sum('valor').alias('total'))
    .orderBy('mes')
)

faturamento_mes.show()

+---+-----+
|mes|total|
+---+-----+
|  1| 5820|
|  2|  950|
|  3| 6790|
+---+-----+



## Desafio Bônus

### Objetivo

Aplicar conceitos de transformação e agregação em um pipeline de dados, além de identificar o tipo de transformação executada em cada etapa.

---

### Desafio

Construa um pipeline que:

1. Filtre pedidos com valor maior que 200  
2. Crie a coluna `valor_anual = valor * 12`  
3. Agrupe os dados por categoria  
4. Calcule a média do valor anual  

---

### Pergunta

Em qual etapa ocorre uma *wide transformation*?

---

### Resposta

A transformação wide ocorre no `groupBy`, pois é necessário redistribuir os dados entre partições para agrupar por categoria, o que gera *shuffle*.  

As etapas anteriores, como `filter` e `withColumn`, são classificadas como *narrow transformations*, pois operam dentro da mesma partição sem necessidade de redistribuição dos dados.

In [46]:
pipeline = (
    df.filter(col('valor') > 200) # Narrow
    .withColumn('valor_anual', col('valor') * 12) # Narrow
    .groupBy('categoria') # Wide
    .agg(avg('valor_anual').alias('media_valor_anual')) # Wide
)

pipeline.show()

+----------+-----------------+
| categoria|media_valor_anual|
+----------+-----------------+
|Eletronico|          29280.0|
|   Calçado|           5700.0|
+----------+-----------------+

